# 02 — Preparando o modelo que vamos monitorar

No [notebook 01](01_eda_home_credit.ipynb), conhecemos os dados e seus problemas.
Agora vamos preparar um modelo de referência: limpar a entrada, separar os dados,
treinar e registrar o resultado no MLflow. Esse mesmo modelo será monitorado no
[notebook 03](03_estrategia_monitoramento.ipynb).

**A pergunta aqui é:** qual desempenho o modelo apresenta antes das mudanças?
`TARGET = 1` representa dificuldade de pagamento. O score ajuda a ordenar os casos;
este projeto não implementa uma decisão de concessão de crédito.

Execute as células de cima para baixo. Este notebook lê a origem e funciona em um
kernel novo, sem depender de variáveis criadas no notebook anterior.

## 1. Quais dados entram no modelo?

Repetimos a amostra estratificada de **20.000 registros, com seed 42**.
O `HomeCreditData` converte idade para anos, flags para `0/1` e remove espaços
nas categorias. O contrato bloqueia entradas inválidas, sem descartar linhas
silenciosamente. Mantemos dez features; o identificador e o alvo ficam fora delas.

A anuidade ausente será preenchida pelo pipeline. Valores extremos válidos são
preservados. Toda a implementação está no `ml_prep_kit`.

In [1]:
import sys
from pathlib import Path

from IPython.display import display
from IPython.utils.io import capture_output

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from ml_prep_kit import (
    DataValidator,
    HomeCreditTrainingWorkflow,
    ModelEvaluator,
    ModelPredictor,
    VisualizationReporter,
)

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_path = project_root / "data/raw/home-credit-default-risk"
processed_path = project_root / "data/processed"
workflow = HomeCreditTrainingWorkflow(
    raw_path,
    processed_path,
    "tech-challenge-4-fintech-training",
    sample_size=20_000,
    random_state=42,
)
preparation = workflow.preparation
validator = DataValidator()
reporter = VisualizationReporter()
evaluator = ModelEvaluator()

clean_df = workflow.load_data()
feature_columns = workflow.feature_columns
print(
    f"Contrato aprovado: {len(clean_df):,} registros e {len(feature_columns)} features."
)
print("Features:", ", ".join(feature_columns))

2026/10/29 19:42:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/29 19:42:59 INFO mlflow.store.db.utils: Updating database tables


Contrato aprovado: 20,000 registros e 10 features.
Features: AGE_YEARS, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, CNT_CHILDREN, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_HOUSING_TYPE, FLAG_OWN_CAR, FLAG_OWN_REALTY


## 2. Como evitamos usar a resposta antes da hora?

Cada conjunto tem uma finalidade. A divisão é estratificada e os IDs não se repetem
entre conjuntos. São recortes aleatórios, sem interpretação de meses reais.

| Conjunto | Parcela | Uso |
|---|---:|---|
| Treino | 60% | Aprender a preparação e o modelo. |
| Validação | 10% | Escolher o limiar que transforma score em alerta. |
| Teste | 10% | Medir o resultado depois das escolhas. |
| Produção simulada | 20% | Reservar clientes para os experimentos de drift. |

In [3]:
partitions, datasets = workflow.split_data(clean_df)
display(
    validator.summarize(partitions)[["dataset", "rows"]].rename(
        columns={"dataset": "Conjunto", "rows": "Registros"}
    )
)
X_train, y_train = datasets["treino"]
X_validation, y_validation = datasets["validacao"]
X_test, y_test = datasets["teste"]

,Conjunto,Registros
0,treino,12000
3,producao,4000
1,validacao,2000
2,teste,2000


## 3. Qual modelo será nossa referência?

Comparamos o **Dummy**, que usa apenas a proporção das classes, com uma
**regressão logística**. O balanceamento dá mais peso à classe minoritária no ajuste.

Usamos o **`sklearn.pipeline.Pipeline`**, criado pelo `ModelFactory` do `ml_prep_kit`.
Ele conecta a preparação ao classificador em um único modelo: as numéricas recebem
mediana e normalização Min-Max; as categóricas, moda e codificação em indicadores;
as binárias são mantidas.
Tudo é aprendido **somente no treino**, uma única vez por modelo. Em produção,
reutilizamos essas regras: não reajustamos a escala para esconder uma mudança.

O MLflow registra o pipeline completo, parâmetros, duração e hash da origem.
A estrutura exibida após o treinamento mostra as etapas reais do scikit-learn.
Contratos e testes de drift são executados sobre os dados antes de reajustar o pipeline.

Os avisos técnicos da execução ficam na saída recolhida da célula; expanda-a para consultá-los.


In [4]:
model_parameters = workflow.model_parameters
tracker = workflow.tracker
tracking_uri = workflow.tracking_uri
source_hash = workflow.source_hash

In [5]:
models, training_records = workflow.train_models()
print("Dummy e regressão logística treinados e registrados no MLflow.")
display(models["logistic_regression"])

2026/10/29 19:43:00 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/10/29 19:43:00 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git branch is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/10/29 19:43:00 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git repository URL is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



/opt/venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/29 19:43:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during des

/opt/venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/29 19:43:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during des

Dummy e regressão logística treinados e registrados no MLflow.


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('bin', 'passthrough',
                                                  ['FLAG_OWN_CAR',
                                                   'FLAG_OWN_REALTY']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('min_max_scaler',
                                                                   MinMaxScaler())]),
                                                  ['AGE_YEARS',
                                                   'AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'CNT_CHILDREN']),
                                                 ('cat',
       

## 4. Qual foi o resultado antes do drift?

Primeiro escolhemos o limiar de maior F1 **na validação**. É um critério técnico
para a demonstração; não representa uma política financeira de concessão de crédito.
Em seguida, congelamos modelo e limiar para avaliar o teste.

In [6]:
validation = evaluator.evaluate_models(
    models,
    {"validacao": (X_validation, y_validation)},
    thresholds={"dummy": 0.5, "logistic_regression": 0.5},
)
selected_threshold, threshold_results = evaluator.select_threshold(
    y_validation, validation["scores"]["logistic_regression"]["validacao"]
)
thresholds = {"dummy": 0.5, "logistic_regression": selected_threshold}
print(f"Limiar escolhido na validação: {selected_threshold:.2f}.")

Limiar escolhido na validação: 0.50.


In [7]:
evaluation = evaluator.evaluate_models(models, datasets, thresholds)
evaluation_reports = evaluation["reports"]
baseline_report = evaluation_reports["logistic_regression"]["teste"]
baseline_result = baseline_report["metrics"]
dummy_result = evaluation_reports["dummy"]["teste"]["metrics"]

A tabela mostra apenas o **teste reservado**:

- **ROC AUC:** capacidade de ordenar os casos; 0,5 é a referência aleatória.
- **Average Precision:** resume precisão e recall; compare com a proporção de positivos.
- **Recall:** parcela dos casos de dificuldade que o modelo identificou no limiar adotado.

A acurácia isolada pode esconder erros na classe minoritária.

In [8]:
comparison = evaluation["global_metrics"].xs("teste", level="split")
display(
    comparison[["roc_auc", "average_precision", "recall"]]
    .rename(
        index={"dummy": "Dummy", "logistic_regression": "Regressão logística"},
        columns={
            "roc_auc": "ROC AUC",
            "average_precision": "Average Precision",
            "recall": "Recall",
        },
    )
    .rename_axis("Modelo")
    .round(3)
)
print(f"Referência para Average Precision: {baseline_result['positive_rate']:.3f}.")

,ROC AUC,Average Precision,Recall
Modelo,,,
Dummy,0.500,0.080,0.000
Regressão logística,0.628,0.116,0.621


Referência para Average Precision: 0.081.


Na matriz, **as linhas são o resultado real e as colunas são a previsão**.
`0` significa ausência de dificuldade e `1`, dificuldade de pagamento.
Um falso negativo é um caso real `1` previsto como `0`; um falso positivo é o inverso.

In [9]:
display(
    baseline_report["confusion_matrix"]
    .rename_axis(index="Real", columns="Previsto")
    .style.background_gradient(cmap="Blues", axis=None)
)
print(
    f"A regressão identificou {baseline_result['true_positives']} de "
    f"{baseline_result['true_positives'] + baseline_result['false_negatives']} "
    f"casos positivos. Gerou {baseline_result['false_positives']} falsos alertas."
)
print(
    f"Diferença de ROC AUC em relação ao Dummy: "
    f"{baseline_result['roc_auc'] - dummy_result['roc_auc']:+.3f}."
)
print(
    f"Brier: regressão {baseline_result['brier_score']:.3f}; "
    f"Dummy {dummy_result['brier_score']:.3f}. Menor é melhor."
)

Previsto,0,1
Real,,
0,1094,745
1,61,100


A regressão identificou 100 de 161 casos positivos. Gerou 745 falsos alertas.
Diferença de ROC AUC em relação ao Dummy: +0.128.
Brier: regressão 0.235; Dummy 0.074. Menor é melhor.


**Como ler o resultado:** ordenar melhor os casos não garante scores calibrados.
O Brier complementa essa leitura, e `class_weight="balanced"` não transforma os
scores automaticamente em probabilidades confiáveis de risco.

Esses valores estabelecem a referência que vamos monitorar. O teste não será usado
para ajustar novamente o modelo; a produção continua reservada para o drift.

## 5. Onde ficam os detalhes da avaliação?

A avaliação completa continua sendo calculada e registrada: métricas por classe e
por conjunto, matrizes, curvas ROC e precisão-recall, calibração, sensibilidade ao
limiar e intervalos de confiança. Para manter a leitura curta, salvamos os gráficos
extras como artefatos, sem repetir todas as saídas aqui.

<details>
<summary>Entender os cuidados da avaliação completa</summary>

Os intervalos de 95% usam 500 reamostragens estratificadas do teste, mantendo modelo,
limiar e proporção das classes fixos. Refletem incerteza amostral condicionada a esse
modelo, sem garantir desempenho futuro. Resultados do treino são descritivos e
podem ser otimistas. A calibração usa dez intervalos e não reajusta o modelo.
Métricas sem denominador ficam indisponíveis; a tabela por classe usa
`zero_division=0`. Entradas futuras podem ultrapassar a escala aprendida e categorias
novas são toleradas pelo encoder, portanto ambas precisam de monitoramento.

</details>

As duas células abaixo geram os detalhes e podem permanecer recolhidas no JupyterLab.

In [10]:
confidence_intervals = evaluator.bootstrap_binary_metrics(
    y_test,
    evaluation["scores"]["logistic_regression"]["teste"],
    threshold=selected_threshold,
    n_resamples=500,
    confidence_level=0.95,
    random_state=preparation.random_state,
)
with capture_output(stdout=False, stderr=False):
    evaluation_figures = {
        "threshold_sensitivity": reporter.plot_thresholds(
            threshold_results, selected_threshold
        ),
        "confusion_matrices": reporter.plot_confusion_matrices(evaluation_reports),
        "roc_precision_recall": reporter.plot_roc_precision_recall(
            evaluation_reports, y_test
        ),
        "calibration_scores": reporter.plot_calibration_scores(
            evaluation, y_test, "logistic_regression", selected_threshold
        ),
    }

In [11]:
evaluation_path = evaluator.save_evaluation(
    evaluation,
    confidence_intervals,
    threshold_results,
    evaluation_figures,
    processed_path / "model/evaluation",
)
for name, reports in evaluation_reports.items():
    tracker.log_evaluation(
        training_records[name]["run_id"],
        reports,
        artifacts=[evaluation_path],
        parameters={"threshold": thresholds[name]},
    )
print("Avaliação completa salva em data/processed/model/evaluation e no MLflow.")

Avaliação completa salva em data/processed/model/evaluation e no MLflow.


Depois de executar, consulte as [métricas completas](../data/processed/model/evaluation/global_metrics.csv),
os [intervalos de confiança](../data/processed/model/evaluation/confidence_intervals.csv),
as [curvas](../data/processed/model/evaluation/roc_precision_recall.png) e a
[calibração](../data/processed/model/evaluation/calibration_scores.png).

## 6. O modelo está pronto para ser monitorado?

Salvamos as partições, o limiar e a identificação do modelo em
`data/processed/experiment.joblib`. O CSV preparado permite inspecionar a base.
Esses arquivos locais ficam fora do Git e conectam este notebook ao próximo.

Antes de seguir, recuperamos o pipeline pelo MLflow e verificamos que ele produz
os mesmos scores. Carregue apenas artefatos confiáveis gerados pelo projeto.

In [12]:
baseline_record = training_records["logistic_regression"]
workflow.save_experiment(
    {
        "threshold": selected_threshold,
        "training_run_id": baseline_record["run_id"],
        "model_uri": baseline_record["model_uri"],
    }
)
loaded = ModelPredictor.from_mlflow(
    baseline_record["model_uri"], tracking_uri=tracking_uri
)
loaded.assert_equivalent(ModelPredictor(models["logistic_regression"]), X_test)
print("Modelo recuperado do MLflow: scores iguais aos do treinamento.")
print(f"Execução de referência: {baseline_record['run_id']}")

Modelo recuperado do MLflow: scores iguais aos do treinamento.
Execução de referência: 9b63d73a14624e2599ea20f3cc72b1e6


**Próxima etapa:** no [notebook 03](03_estrategia_monitoramento.ipynb), recuperaremos
este modelo, criaremos a base v2 e usaremos histogramas sobrepostos para entender
onde os dados mudaram. Também veremos um lote inválido e uma mudança de desfecho
que o monitoramento das entradas, sozinho, não identifica.

Temos um baseline rastreável para a demonstração. O [README](../README.md) explica
a governança, os vieses encontrados e o workflow de retreino, aprovação e rollback.
A aprovação automática usa critérios técnicos; não autoriza decisões reais de crédito.